<a href="https://colab.research.google.com/github/MM24-6/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from huggingface_hub import login
login()

In [9]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [10]:
import itertools
import pandas as pd

rows = list(itertools.islice(ds, 50000))
df = pd.DataFrame(rows)

print(df.shape)
df.head()

(50000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule:

Pages with high impressions but low clicks will receive a higher baseline score because they have the greatest opportunity for improvement.

Reason Code:

LOW_CTR_HIGH_IMPRESSIONS – The page receives many impressions but relatively few clicks, so improving its title or content may increase traffic.

Action:

Review and optimize the page for better click-through performance.

---

Signal Check 1

Signal: Impressions vs CTR

Verdict: OPPOSITE

The bucket table shows that pages with fewer impressions have a higher average CTR, while pages with more impressions have a lower average CTR. This is the opposite of the expected relationship. The bucket table and n values were checked before defining the rule.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)) * 100

signal = (
    df.groupby(pd.cut(df["gsc_impressions"], bins=[0,10,50,100,1000]))
      .agg(
          n=("gsc_impressions","count"),
          avg_ctr=("ctr","mean")
      )
)

print(signal)

                     n   avg_ctr
gsc_impressions                 
(0, 10]          29333  0.830664
(10, 50]         18003  0.669370
(50, 100]         2022  0.663173
(100, 1000]        642  0.606456


/tmp/ipykernel_457/1450060387.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df["gsc_impressions"], bins=[0,10,50,100,1000]))


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I created a simple baseline scoring rule based on search impressions and CTR. Pages with higher impressions and lower CTR receive a higher score because they have a greater opportunity for optimization. The notebook ranks all pages, assigns one reason code and one action label, and writes the ranked queue to baseline_action_score.csv.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)) * 100

df["baseline_score"] = df["gsc_impressions"] - (df["ctr"] * 2)

df["reason_code"] = "LOW_CTR_HIGH_IMPRESSIONS"
df["action"] = "Review and optimize page"

ranked = df.sort_values("baseline_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(ranked[["gsc_impressions","gsc_clicks","ctr","baseline_score","reason_code","action"]].head(10))

print("CSV written successfully.")

       gsc_impressions  gsc_clicks       ctr  baseline_score  \
38579              818          13  1.589242      814.821516   
42839              742          16  2.156334      737.687332   
47089              714          13  1.820728      710.358543   
34266              590           8  1.355932      587.288136   
14715              580           8  1.379310      577.241379   
12108              535           5  0.934579      533.130841   
21286              534          15  2.808989      528.382022   
12435              532          11  2.067669      527.864662   
41328              503           0  0.000000      503.000000   
6854               506          11  2.173913      501.652174   

                    reason_code                    action  
38579  LOW_CTR_HIGH_IMPRESSIONS  Review and optimize page  
42839  LOW_CTR_HIGH_IMPRESSIONS  Review and optimize page  
47089  LOW_CTR_HIGH_IMPRESSIONS  Review and optimize page  
34266  LOW_CTR_HIGH_IMPRESSIONS  Review and optimize pa

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top-ranked pages have high impressions but relatively low CTR, so they are good candidates for optimization. Confidence is moderate because this baseline uses only simple search signals. The ranking could be wrong if the page already performs well for its purpose, has seasonal traffic, or requires no further optimization.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked.head(20).copy()

top20["confidence_note"] = "Medium"
top20["what_would_make_it_wrong"] = (
    "Seasonal traffic, branded queries, or already optimized content."
)

print(
    top20[
        [
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)

                         action               reason_code confidence_note  \
38579  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
42839  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
47089  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
34266  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
14715  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
12108  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
21286  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
12435  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
41328  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
6854   Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
40553  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   
20971  Review and optimize page  LOW_CTR_HIGH_IMPRESSIONS          Medium   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some ranked pages may not actually need optimization because of seasonal trends or branded searches. I did not use future information, labels, or FlyRank product flags in my scoring rule, so no data leakage was introduced.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Top 5 weakest picks:")
print(ranked.tail(5)[["gsc_impressions", "gsc_clicks", "ctr", "baseline_score"]])

print("\nLeakage Check")
print("No future-window features used.")
print("No product flags used.")
print("Only current search signals were used.")

Top 5 weakest picks:
       gsc_impressions  gsc_clicks    ctr  baseline_score
29528                1           1  100.0          -199.0
8979                 1           1  100.0          -199.0
48532                1           1  100.0          -199.0
1460                 1           1  100.0          -199.0
26442                1           1  100.0          -199.0

Leakage Check
No future-window features used.
No product flags used.
Only current search signals were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


✅ Every section above is filled — markdown thinking AND the code that backs it.

✅ The notebook runs top to bottom with no errors (Runtime → Run all).

✅ No client names, URLs, or private queries anywhere.

✅ My claims use careful words: observed, measured, directional, decision-support.

✅ Committed to my repo under work/notebooks/ and ready to submit.